[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aaxhirrr/mat-422-fall-2026/blob/AASHIR/hw3-7/HW3.7/HW3_7_neural_networks.ipynb)

# homework 3.7: artificial neural networks

Aashir Javed | MAT 422

A neural network takes an input through a few layers of weighted sums and nonlinear functions. We can see what each part does with a small example instead of a large dataset. This notebook covers the mathematical formulation, activation functions, cost functions, backpropagation, and the training algorithm.

I use the four inputs of XOR: the label is 1 when the two inputs differ and 0 when they match. The positive and negative points sit on opposite diagonals, so a single straight boundary cannot separate them. A hidden layer gives us a way to learn that pattern.

## setup

We only need NumPy and Matplotlib. All inputs and starting weights are written out, so the results are repeatable without downloading data. **Runtime > Run all** in Colab runs the notebook from top to bottom.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=5, suppress=True)
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y = np.array([[0], [1], [1], [0]], dtype=float)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

print("input 1, input 2, XOR label")
print(np.column_stack([X, y]).astype(int))

## 3.7.1 mathematical formulation

The network has two inputs, four hidden units, and one output. For a single unit, we first form $z=sum_j w_jx_j+b$, then apply an activation $a=g(z)$. The weights control each input's contribution, and the bias shifts the weighted sum.

For a whole layer, the same idea becomes a matrix product. Using observations as rows, our forward pass is

$$Z_1=XW_1+b_1,\qquad A_1=\tanh(Z_1),$$
$$Z_2=A_1W_2+b_2,\qquad P=\sigma(Z_2).$$

Here $\sigma(z)=1/(1+e^{-z})$, and $P$ contains the model's predicted probabilities for class 1. We predict class 1 when $P\geq 0.5$. The bias row is added to every observation.

| quantity | shape | meaning |
|---|---|---|
| $X$ | $4\times 2$ | four input pairs |
| $W_1$, $b_1$ | $2\times 4$, $1\times 4$ | hidden-layer weights and biases |
| $A_1$ | $4\times 4$ | hidden activations |
| $W_2$, $b_2$ | $4\times 1$, $1\times 1$ | output weights and bias |
| $P$, $y$ | $4\times 1$ | probabilities and labels |

That gives $8+4+4+1=17$ trainable numbers. The starting weights below are different across hidden units. Identical starting units would receive identical updates and keep learning the same feature.

In [ ]:
initial = {
    "W1": np.array([[0.4, -0.6, 0.2, 0.5], [-0.3, 0.8, -0.5, 0.1]]),
    "b1": np.array([[0.1, 0.0, -0.1, 0.2]]),
    "W2": np.array([[0.7], [-0.4], [0.6], [-0.2]]),
    "b2": np.array([[0.05]])
}

def forward(X, params):
    z1 = X @ params["W1"] + params["b1"]
    a1 = np.tanh(z1)
    z2 = a1 @ params["W2"] + params["b2"]
    probability = sigmoid(z2)
    return z1, a1, z2, probability

z1, a1, z2, initial_prob = forward(X, initial)
print("hidden weighted sums for input [0, 1]:", z1[1])
print("hidden activations for that input:", a1[1])
print("output weighted sum:", z2[1, 0])
print("probability of class 1:", initial_prob[1, 0])
print("all initial probabilities:", initial_prob.ravel())
print("number of parameters:", sum(value.size for value in initial.values()))

# This is the same calculation for just the first hidden unit.
one_hidden_sum = 0 * 0.4 + 1 * (-0.3) + 0.1
print("first hidden sum checked by hand:", one_hidden_sum)

For input $[0,1]$, the first hidden weighted sum is $0(0.4)+1(-0.3)+0.1=-0.2$. The four hidden sums are $[-0.2,0.8,-0.6,0.3]$. After the activations and output layer, the probability is about $0.3243$, even though the correct label is 1. The network can produce an answer already, but its initial answer needs improvement.

## 3.7.2 activation functions

The hidden activation matters because stacking affine layers without a nonlinearity still gives one affine map:

$$ (XW_1+b_1)W_2+b_2=X(W_1W_2)+(b_1W_2+b_2). $$

That would leave a straight decision boundary for our binary classifier. We can compare three common nonlinear choices and the derivatives used during backpropagation:

| activation | function | derivative |
|---|---|---|
| sigmoid | $\sigma(z)=1/(1+e^{-z})$ | $\sigma(z)(1-\sigma(z))$ |
| tanh | $\tanh(z)$ | $1-\tanh^2(z)$ |
| ReLU | $\max(0,z)$ | $0$ for $z<0$, $1$ for $z>0$ |

ReLU is not differentiable at zero; training software normally chooses a value there, often zero. The gap in its derivative plot marks that exception. Our hidden layer uses tanh, and our output uses sigmoid to keep probabilities between 0 and 1.

In [ ]:
z = np.linspace(-5, 5, 401)
activations = [sigmoid(z), np.tanh(z), np.maximum(0, z)]
relu_derivative = (z > 0).astype(float)
relu_derivative[z == 0] = np.nan
derivatives = [sigmoid(z) * (1 - sigmoid(z)), 1 - np.tanh(z)**2, relu_derivative]
names = ["sigmoid", "tanh", "ReLU"]

fig, axes = plt.subplots(2, 3, figsize=(11, 5.4), sharex=True)
for j, name in enumerate(names):
    axes[0, j].plot(z, activations[j])
    axes[0, j].set(title=name, ylabel="activation")
    axes[1, j].plot(z, derivatives[j], color="tab:orange")
    axes[1, j].set(xlabel="weighted sum z", ylabel="derivative")
    axes[1, j].set_ylim(-0.08, 1.08)
    for ax in axes[:, j]:
        ax.axvline(0, color="gray", linewidth=0.6)
        ax.grid(alpha=0.2)
axes[1, 2].plot([0, 0], [0, 1], "o", markerfacecolor="white", color="tab:orange")
fig.tight_layout()
plt.show()

linear_hidden_output = (X @ initial["W1"] + initial["b1"]) @ initial["W2"] + initial["b2"]
collapsed_output = X @ (initial["W1"] @ initial["W2"]) + initial["b1"] @ initial["W2"] + initial["b2"]
print("two affine layers collapse to one:", np.allclose(linear_hidden_output, collapsed_output))

Sigmoid and tanh flatten out at large positive or negative inputs, and their derivatives approach zero there. Multiplying many small derivatives can make earlier layers learn slowly. ReLU keeps derivative 1 on its positive side, but its negative side has derivative 0. None of these choices fixes every training problem. For this small network, tanh gives a smooth hidden transformation that is easy to differentiate.

## 3.7.3 cost function

The cost combines the prediction errors into one number. For numeric targets, a common choice is mean squared error:

$$J_{\mathrm{MSE}}=\frac{1}{n}\sum_{i=1}^n(p_i-y_i)^2.$$

For our 0/1 labels, we train with mean binary cross-entropy:

$$J=-\frac{1}{n}\sum_{i=1}^n\left[y_i\log p_i+(1-y_i)\log(1-p_i)\right].$$

This is the average negative log-likelihood for Bernoulli labels. A correct, confident prediction has a small loss; a confidently wrong prediction has a large loss. The two costs have different scales, so a smaller numerical value across different cost definitions does not tell us which model is better.

With $p_i=\sigma(z_i)$, we can write the same cross-entropy as

$$J=\frac{1}{n}\sum_i\left[\log(1+e^{z_i})-y_i z_i\right].$$

The code uses `np.logaddexp(0, z)` for $\log(1+e^z)$. That keeps the calculation stable without taking the log of a probability rounded to zero.

In [ ]:
def cross_entropy(y_true, logits):
    return np.mean(np.logaddexp(0, logits) - y_true * logits)

initial_loss = cross_entropy(y, z2)
probability_loss = -np.mean(y * np.log(initial_prob) + (1 - y) * np.log(1 - initial_prob))
print(f"initial cross-entropy: {initial_loss:.6f}")
print(f"initial mean squared error: {np.mean((initial_prob - y)**2):.6f}")
print("the two cross-entropy formulas agree:", np.isclose(initial_loss, probability_loss))
print(f"loss for y=1, p=0.9: {-np.log(0.9):.4f}")
print(f"loss for y=1, p=0.1: {-np.log(0.1):.4f}")

p_values = np.linspace(0.01, 0.99, 200)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
for label, color in [(0, "tab:blue"), (1, "tab:orange")]:
    bce = -(label * np.log(p_values) + (1 - label) * np.log(1 - p_values))
    axes[0].plot(p_values, bce, color=color, label=f"true label {label}")
    axes[1].plot(p_values, (p_values - label)**2, color=color, label=f"true label {label}")
for ax, title in zip(axes, ["Binary cross-entropy", "Squared error"]):
    ax.set(xlabel="predicted probability of class 1", ylabel="loss for one observation", title=title)
    ax.legend()
fig.tight_layout()
plt.show()

The initial cross-entropy is about $0.7172$. For a positive label, changing the probability from $0.9$ to $0.1$ raises its loss from about $0.1054$ to $2.3026$. The left plot makes that penalty visible. We will minimize this cross-entropy during training; the squared-error plot is a comparison of cost functions.

## 3.7.4 backpropagation

Backpropagation applies the chain rule from the output back toward the inputs. It computes the gradients; gradient descent uses those gradients to change the weights. These are two separate steps.

For one observation with a sigmoid output and cross-entropy,

$$\frac{\partial \ell}{\partial z}=\left(-\frac{y}{p}+\frac{1-y}{1-p}\right)p(1-p)=p-y.$$

Since our cost is a mean, the output-layer error matrix is $D_2=(P-y)/n$. For the row-based arrays above, the remaining derivatives are

$$\frac{\partial J}{\partial W_2}=A_1^T D_2,\qquad
\frac{\partial J}{\partial b_2}=\sum_{i=1}^n D_{2,i:},$$
$$D_1=(D_2W_2^T)\odot(1-A_1^2),$$
$$\frac{\partial J}{\partial W_1}=X^T D_1,\qquad
\frac{\partial J}{\partial b_1}=\sum_{i=1}^n D_{1,i:}.$$

The symbol $\odot$ means elementwise multiplication. The factor $1-A_1^2$ comes from the tanh derivative. We divide by $n$ once, in $D_2$, and keep the old weights until every gradient has been computed.

In [ ]:
def gradients(X, y, params):
    z1, a1, z2, probability = forward(X, params)
    dz2 = (probability - y) / len(X)
    dz1 = (dz2 @ params["W2"].T) * (1 - a1**2)
    return {
        "W1": X.T @ dz1,
        "b1": dz1.sum(axis=0, keepdims=True),
        "W2": a1.T @ dz2,
        "b2": dz2.sum(axis=0, keepdims=True)
    }

initial_gradients = gradients(X, y, initial)
for name, grad in initial_gradients.items():
    print(f"gradient for {name}, shape {grad.shape}:")
    print(grad)

# A small step should move the cost down at this starting point.
trial_rate = 0.1
trial = {name: value - trial_rate * initial_gradients[name] for name, value in initial.items()}
trial_loss = cross_entropy(y, forward(X, trial)[2])
print(f"cost before the step: {initial_loss:.6f}")
print(f"cost after the step:  {trial_loss:.6f}")

For example, the gradient of the first weight in $W_1$ is about $0.02249$. Subtracting $0.1$ times that gradient moves its value from $0.4$ to about $0.39775$. The trial step lowers the total cost. A large step does not automatically do that, so the learning rate still matters.

### checking the chain rule numerically

I also check all 17 derivatives without using the backpropagation formulas. For a parameter $\theta_j$, the centered difference is

$$\frac{\partial J}{\partial\theta_j}\approx
\frac{J(\theta_j+h)-J(\theta_j-h)}{2h}.$$

We change one number at a time, keep the others fixed, and compare with the analytical gradient. This is useful for checking the implementation, but it would be expensive as a training method because it needs two forward passes per parameter.

In [ ]:
h = 1e-5
gradient_errors = {}
checked_parameters = 0

for name, values in initial.items():
    numeric = np.zeros_like(values)
    for index in np.ndindex(values.shape):
        plus = {key: value.copy() for key, value in initial.items()}
        minus = {key: value.copy() for key, value in initial.items()}
        plus[name][index] += h
        minus[name][index] -= h
        loss_plus = cross_entropy(y, forward(X, plus)[2])
        loss_minus = cross_entropy(y, forward(X, minus)[2])
        numeric[index] = (loss_plus - loss_minus) / (2 * h)
        checked_parameters += 1
    gradient_errors[name] = np.max(np.abs(numeric - initial_gradients[name]))
    print(f"{name}: largest absolute gradient difference = {gradient_errors[name]:.2e}")

print("parameters checked:", checked_parameters)
assert max(gradient_errors.values()) < 1e-7

Each parameter block passes the $10^{-7}$ absolute-error tolerance. That checks the signs, transposes, activation derivative, and averaging factor together. It does not prove that gradient descent will find the best possible network, but it gives us a separate check of the derivatives we are about to use.

## 3.7.5 backpropagation algorithm

Now we can repeat the whole process:

1. Run the forward pass for the four observations.
2. Compute the cost and its gradients using backpropagation.
3. Update every parameter with $\theta\leftarrow\theta-\eta\nabla J(\theta)$.
4. Repeat with the updated parameters.

Here $\eta=0.5$, and one step uses the entire four-row dataset. This is full-batch gradient descent, with 4,000 steps and no automatic differentiation library. The nonlinear network objective is generally nonconvex, so convergence to a global minimum is not guaranteed. Different starting weights and step sizes can give different results.

The training code starts from a copy of the original weights, so rerunning this cell gives the same training run. The final printed cost is measured after the last update.

In [ ]:
params = {name: value.copy() for name, value in initial.items()}
learning_rate = 0.5
steps = 4000
losses = [cross_entropy(y, forward(X, params)[2])]

for step in range(steps):
    grads = gradients(X, y, params)
    for name in params:
        params[name] -= learning_rate * grads[name]
    losses.append(cross_entropy(y, forward(X, params)[2]))

final_prob = forward(X, params)[3]
predicted = (final_prob >= 0.5).astype(int)
print(f"initial cost: {losses[0]:.6f}")
print(f"final cost:   {losses[-1]:.6f}")
print("input       label   probability   prediction")
for inputs, target, prob, prediction in zip(X, y.ravel(), final_prob.ravel(), predicted.ravel()):
    print(f"{inputs.astype(int)}        {int(target)}       {prob:.5f}          {prediction}")
print(f"accuracy on the four XOR inputs: {np.mean(predicted == y):.0%}")

xx, yy = np.meshgrid(np.linspace(-0.25, 1.25, 180), np.linspace(-0.25, 1.25, 180))
grid = np.column_stack([xx.ravel(), yy.ravel()])
grid_prob = forward(grid, params)[3].reshape(xx.shape)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].plot(np.arange(steps + 1), losses)
axes[0].set(xlabel="completed update steps", ylabel="mean binary cross-entropy", title="Training cost")
surface = axes[1].contourf(xx, yy, grid_prob, levels=np.linspace(0, 1, 11), cmap="coolwarm", alpha=0.8)
axes[1].contour(xx, yy, grid_prob, levels=[0.5], colors="black", linewidths=1.5)
for label, marker in [(0, "o"), (1, "^")]:
    selected = y.ravel() == label
    axes[1].scatter(X[selected, 0], X[selected, 1], marker=marker, s=100,
                    facecolors="white", edgecolors="black", label=f"label {label}")
axes[1].set(xlabel="input 1", ylabel="input 2", title="XOR probabilities and the 0.5 boundary", aspect="equal")
axes[1].legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2, fontsize=8)
fig.colorbar(surface, ax=axes[1], label="probability of class 1")
fig.tight_layout()
plt.show()

The cost falls from about $0.7172$ to $0.001238$. The final probabilities, in input order, are approximately $[0.00016,0.99845,0.99860,0.00184]$. Thresholding at $0.5$ gives $[0,1,1,0]$, which matches all four XOR labels. The boundary bends to separate the diagonal groups, something a single affine classifier could not do.

This is a fit to the complete XOR truth table, not an accuracy estimate on unseen data. The colors between the four points show the network's interpolation, not extra labeled observations. For a larger real dataset, we would keep validation and test observations separate and check for overfitting.

## checking the results

We can finish by checking the shapes, the hand calculation, the cost, the gradients, and the learned labels. These checks are small enough to read alongside the examples.

In [ ]:
checks = {
    "forward-pass shapes match the formulation": z1.shape == (4, 4) and a1.shape == (4, 4)
        and z2.shape == (4, 1) and initial_prob.shape == (4, 1),
    "first hidden sum agrees with the hand calculation": np.isclose(z1[1, 0], one_hidden_sum),
    "all 17 parameters were checked": checked_parameters == 17,
    "two affine layers collapse to one": np.allclose(linear_hidden_output, collapsed_output),
    "sigmoid and tanh values at zero are correct": np.isclose(sigmoid(0), 0.5) and np.isclose(np.tanh(0), 0),
    "cross-entropy formulas agree": np.isclose(initial_loss, probability_loss),
    "gradient arrays match their parameter shapes": all(initial_gradients[name].shape == initial[name].shape for name in initial),
    "backpropagation matches finite differences": max(gradient_errors.values()) < 1e-7,
    "the small trial step lowers the cost": trial_loss < initial_loss,
    "the full training history is finite": np.all(np.isfinite(losses)),
    "training lowers the cost": losses[-1] < losses[0],
    "saved final loss matches the final weights": np.isclose(losses[-1], cross_entropy(y, forward(X, params)[2])),
    "final probabilities stay between zero and one": np.all((final_prob > 0) & (final_prob < 1)),
    "all four XOR labels are correct": np.array_equal(predicted, y)
}
for label, passed in checks.items():
    print(f"{label}: {bool(passed)}")
assert all(checks.values()), "one of the checks needs another look"

## what I took from this

The forward pass is mostly matrix multiplication, but the activation functions make the network more than one affine map. I found it useful to connect the derivative formulas to the array shapes, especially the transpose in each weight gradient. The finite-difference check also gives us a way to catch a mistake before trusting the training curve.

Backpropagation supplies the gradients, and the repeated gradient-descent updates turn those gradients into learning. Getting all four XOR inputs right shows that this small network can represent the pattern. It does not settle how well a larger model would work on new observations.

All five assigned topics have formulas, code, and explanations of the output. Before submitting, run the notebook from the top, review the results, and check that the Open in Colab button works from GitHub. The starting weights and learning rate are easy to change to see how the training responds. The rubric also asks for saving the notebook directly from Colab to GitHub.

## references

- [MAT 422 section 3.7 guidance](https://github.com/DynamicLLM/MAT422/blob/main/Skills/Student-Home-Work-Skills-Pack/mat422-section-3-7-neural-networks/references/section-3-7-guidance.md)
- [Goodfellow, Bengio, and Courville, Deep Learning, chapter 6](https://www.deeplearningbook.org/contents/mlp.html)
- [NumPy: logaddexp](https://numpy.org/doc/stable/reference/generated/numpy.logaddexp.html)

- AI use note: I had help from Codex in refining the code (not completely writing it), refining the latex of the report, and pushing it to GitHub. Other than that, the intial psuedocode/boilerplates, logic, and the report is still drafted by me. I ran every cell and reviewed the math and explanations before submitting.